# Many Models Forecasting Demo (FoundationForecast models, weekly)
This notebook runs the foundation models MMF supports through [foundationforecast](https://pypi.org/project/foundationforecast/) on the weekly series of the [M4 competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128#sec5). The same notebook runs on a classic GPU cluster and on serverless GPU compute.

These models are univariate, and inference always runs on the driver's GPU, so one GPU processes all series. See [mmf_sa/models/README.md](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/README.md#foundationforecast) for the model list, licenses and hyperparameters.

### Compute setup

Choose one of the following:

- **Classic cluster:** a single-node cluster with one A10G GPU (for example `g5.xlarge` on AWS) running [Databricks Runtime 18 for ML](https://docs.databricks.com/aws/en/release-notes/runtime/18ml). Extra GPUs are not used. Leave the `serverless` widget set to `false`.
- **Serverless GPU:** attach the notebook to [serverless compute](https://docs.databricks.com/aws/en/compute/serverless/notebooks), choose `A10` as the [accelerator](https://docs.databricks.com/aws/en/compute/serverless/gpu), and set the [environment version](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to Standard v6. Set the `serverless` widget to `true`.

### Install packages
The cell below installs MMF from this repository with the `foundationforecast` extra, which pins the same packages as [requirements-foundationforecast.txt](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-foundationforecast.txt). Use a fresh environment: don't combine it with `requirements-foundation.txt`.

In [0]:
%pip install "../..[foundationforecast]" datasetsforecast==0.0.8 --quiet
dbutils.library.restartPython()

### Hugging Face token (optional)
None of the checkpoints used here is gated, so no token is required. If you hit anonymous rate limits on the Hugging Face Hub, store a token in a [Databricks secret](https://docs.databricks.com/aws/en/security/secrets/) and uncomment the cell below. Never paste the token into the notebook or a config file.

In [0]:
# import os
# os.environ["HF_TOKEN"] = dbutils.secrets.get(scope="<secret-scope>", key="<secret-key>")

In [0]:
import logging
import uuid
import pathlib
import pandas as pd
from datasetsforecast.m4 import M4

logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

### Parameters
Provide the catalog and schema where the data, the outputs and the registered models are stored, and whether the notebook runs on serverless GPU.

In [0]:
dbutils.widgets.text("catalog", "mmf")
dbutils.widgets.text("db", "m4")
dbutils.widgets.dropdown("serverless", "false", ["false", "true"])

catalog = dbutils.widgets.get("catalog")  # Name of the catalog we use to manage our assets
db = dbutils.widgets.get("db")  # Name of the schema we use to manage our assets (e.g. datasets)
serverless = dbutils.widgets.get("serverless") == "true"
user = spark.sql("select current_user() as user").collect()[0]["user"]  # User email address

### Prepare data
We use the [`datasetsforecast`](https://github.com/Nixtla/datasetsforecast/tree/main/) package to download the M4 data and convert the weekly series to the format MMF expects. All series end on the same date: MMF's data quality check removes series whose dates stop before the latest date in the dataset.

In [0]:
# Number of time series
n = 1000  # M4 contains 359 weekly series, so all of them are used


def create_m4_weekly():
    y_df, _, _ = M4.load(directory=str(pathlib.Path.home()), group="Weekly")
    target_ids = {f"W{i}" for i in range(1, n + 1)}
    y_df = y_df[y_df["unique_id"].isin(target_ids)]
    y_df = (
        y_df.groupby("unique_id", group_keys=False)
             .apply(lambda g: transform_group(g, g.name))
             .reset_index(drop=True)
    )
    return y_df


def transform_group(df, unique_id):
    if len(df) > 260:
        df = df.iloc[-260:]
    end = pd.Timestamp("2024-12-29")
    date_idx = pd.date_range(end=end, periods=len(df), freq="W-SUN", name="ds")
    res_df = pd.DataFrame({
        "ds": date_idx,
        "unique_id": unique_id,
        "y": df["y"].to_numpy()
    })
    return res_df

In [0]:
# Making sure that the catalog and the schema exist
if catalog not in [row[0] for row in spark.sql("SHOW CATALOGS").collect()]:
    _ = spark.sql(f"CREATE CATALOG {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

(
    spark.createDataFrame(create_m4_weekly())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.ff_weekly_train")
)

Let's take a peek at the dataset:

In [0]:
display(spark.sql(f"select count(distinct unique_id) as series, min(ds) as first_date, max(ds) as last_date from {catalog}.{db}.ff_weekly_train"))

In [0]:
display(
  spark.sql(f"select * from {catalog}.{db}.ff_weekly_train where unique_id in ('W1', 'W2', 'W3', 'W4', 'W5') order by unique_id, ds")
  )

Weekly forecasting requires the timestamp column to fall on a Sunday, the end of the ISO week.

### Models
The FoundationForecast models start with `FF`. Before you run them, note:

- `FFTimesFM_3_0` and `FFPatchTSTFM_R1` are released under non-commercial licenses. MMF logs a warning the first time each one runs. Remove them from the list if that matters for your use case.
- Covariate columns are ignored: these models forecast from the target history only.

You can change the hyperparameters in [mmf_sa/models/models_conf.yaml](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/models_conf.yaml).

In [0]:
active_models = [
    "FFChronos2",
    "FFChronos2Small",
    "FFTimesFM_2_5_200m",
    "FFTimesFM_3_0",
    "FFTiRex2",
    "FFToto",
    "FFToto2_22m",
    "FFToto2_313m",
    "FFToto2_1B",
    "FFFlowState",
    "FFPatchTSTFM_R2",
    "FFPatchTSTFM_R1",
    "FFTafsutBase",
    "FFT0Beta",
]

### Run MMF

Now we run evaluation and forecasting with the `run_forecast` function. Refer to [README.md](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/README.md#parameters-description) for a description of each parameter. Each model is backtested, logged and registered to Unity Catalog under `model_output`, and then used to forecast. The same `run_id` is assigned to all models, which makes the post-evaluation analysis easier. On the first run, each checkpoint is downloaded from the Hugging Face Hub.

In [0]:
from mmf_sa import run_forecast

run_id = str(uuid.uuid4())

for model in active_models:
    print(f"\n===== Running {model} (run_id={run_id}) =====")
    run_forecast(
        spark=spark,
        train_data=f"{catalog}.{db}.ff_weekly_train",
        scoring_data=f"{catalog}.{db}.ff_weekly_train",
        scoring_output=f"{catalog}.{db}.ff_weekly_scoring_output",
        evaluation_output=f"{catalog}.{db}.ff_weekly_evaluation_output",
        model_output=f"{catalog}.{db}",
        group_id="unique_id",
        date_col="ds",
        target="y",
        freq="W",
        prediction_length=4,
        backtest_length=12,
        stride=1,
        metric="smape",
        train_predict_ratio=1,
        data_quality_check=True,
        resample=False,
        active_models=[model],
        experiment_path=f"/Users/{user}/mmf/foundationforecast_weekly",
        use_case_name="ff_weekly",
        run_id=run_id,
        accelerator="gpu",
        serverless=serverless,
    )

### Evaluate
The `evaluation_output` table stores the results of all backtesting trials for every series and model.

In [0]:
display(spark.sql(f"""
    select * from {catalog}.{db}.ff_weekly_evaluation_output
    where unique_id = 'W1' and run_id = '{run_id}'
    order by unique_id, model, backtest_window_start_date
    """))

In [0]:
display(spark.sql(f"""
    select model, avg(metric_value) as smape
    from {catalog}.{db}.ff_weekly_evaluation_output
    where run_id = '{run_id}'
    group by model
    order by smape
    """))

### Forecast
The `scoring_output` table stores the forecasts for each series from each model.

In [0]:
display(spark.sql(f"""
    select * from {catalog}.{db}.ff_weekly_scoring_output
    where unique_id = 'W1' and run_id = '{run_id}'
    order by unique_id, model
    """))

Refer to the [notebook](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/post-evaluation-analysis.ipynb) for guidance on performing fine-grained model selection after running `run_forecast`.

### Delete Tables
Let's clean up the tables.

In [0]:
#display(spark.sql(f"delete from {catalog}.{db}.ff_weekly_evaluation_output"))

In [0]:
#display(spark.sql(f"delete from {catalog}.{db}.ff_weekly_scoring_output"))